# Tratamento e preparação dos dados

## Objetivo

Aplicar as regras de tratamento definidas na etapa de identificação de inconsistências, preservando os arquivos originais da camada Bronze e gerando dados preparados para a camada Silver.

## Estrutura do notebook

1. Importação e leitura dos dados da camada Bronze
2. Criação de cópias para tratamento
3. Padronização de valores categóricos
4. Tratamento de datas e valores ausentes
5. Tratamento de integridade referencial
6. Validação dos dados tratados
7. Salvamento na camada Silver

In [1]:
import pandas as pd
import pyarrow

ativos = pd.read_csv('../data/bronze/ativos.csv', sep=';')
vulnerabilidades = pd.read_csv('../data/bronze/vulnerabilidades.csv', sep=';')


## 2. Criação de cópias para tratamento

In [2]:
ativos_tratados = ativos.copy()
vulnerabilidades_tratadas = vulnerabilidades.copy()

## 3. Padronização de valores categóricos


In [3]:
mapeamento_vulnerabilidades_severidade = {
    'media': 'Média',
    'BaixA': 'Baixa',
    'ALTA': 'Alta',
    'critica': 'Crítica',
    'Critica': 'Crítica',
    'Media': 'Média',
    'Alto': 'Alta',
}

vulnerabilidades_tratadas['severidade'] = vulnerabilidades_tratadas['severidade'].replace(mapeamento_vulnerabilidades_severidade)
vulnerabilidades_tratadas['severidade'].value_counts(dropna=False)

severidade
Média      506
Crítica    501
Alta       488
Baixa      479
NaN         26
Name: count, dtype: int64

In [4]:
mapeamento_vulnerabilidades_origem = {
    'CloudSecurity': 'Cloud Security',
    'Cloud-Security': 'Cloud Security',
}

vulnerabilidades_tratadas['origem'] = vulnerabilidades_tratadas['origem'].replace(mapeamento_vulnerabilidades_origem)
vulnerabilidades_tratadas['origem'].value_counts(dropna=False)

origem
Veracode          522
Pentest           507
Cloud Security    506
Qualys            465
Name: count, dtype: int64

In [5]:
mapeamento_ativos_criticidade = {
    'media': 'Média',
    'BaixA': 'Baixa',
    'critica': 'Crítica',
    ' ': pd.NA,
    'NaN': pd.NA
}

ativos_tratados['criticidade'] = ativos_tratados['criticidade'].replace(mapeamento_ativos_criticidade)
ativos_tratados['criticidade'].value_counts(dropna=False)

criticidade
Baixa      58
Média      50
Crítica    46
Alta       42
NaN         4
Name: count, dtype: int64

Após a aplicação dos mapeamentos, as categorias foram consolidadas em uma representação única.

## 4. Tratamento de datas

### Tratamento de `data_abertura`

In [6]:
vulnerabilidades_tratadas['data_abertura'] = pd.to_datetime(
    vulnerabilidades_tratadas['data_abertura'],
    format='%d/%m/%Y',
    errors='coerce'
)

vulnerabilidades_tratadas['data_abertura'].isna().sum()

np.int64(78)

In [7]:
vulnerabilidades_tratadas['data_abertura'] = vulnerabilidades_tratadas['data_abertura'].fillna(
    pd.to_datetime(
        vulnerabilidades['data_abertura'],
        format='%m-%d-%Y',
        errors='coerce'
    )
)
vulnerabilidades_tratadas['data_abertura'].isna().sum()

np.int64(50)

In [8]:
vulnerabilidades_tratadas['data_abertura'] = vulnerabilidades_tratadas['data_abertura'].fillna(
    pd.to_datetime(
        vulnerabilidades['data_abertura'],
        format='%Y-%d-%m',
        errors='coerce'
    )
)

vulnerabilidades_tratadas['data_abertura'].isna().sum()

np.int64(22)

Os registros com datas de abertura que não representam uma data válida foram mantidos com `NaT`.


### Tratamento de `data_correcao`


In [9]:
vulnerabilidades_tratadas['data_correcao'] = pd.to_datetime(
    vulnerabilidades_tratadas['data_correcao'],
    format='%d/%m/%Y',
    errors='coerce'
)

vulnerabilidades_tratadas['data_correcao'].isna().sum()

np.int64(1531)

Como a ausencia de valores em ´data_correcao´ já foi validade e é esperada, foi apenas transformado os valores no formato data

## 5. Tratamento de vulnerabilidade associada a ativo inexistente

In [10]:
vulnerabilidades_tratadas = vulnerabilidades_tratadas[vulnerabilidades_tratadas['ativo_id'] != 'A9999']
vulnerabilidades = vulnerabilidades[vulnerabilidades['ativo_id'] == 'A9999']
vulnerabilidades

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
100,V00101,A9999,Média,Cloud Security,08/06/2026,NaN,Aceita,3.9
500,V00501,A9999,Crítica,Veracode,07/05/2026,NaN,Em tratamento,4.7
900,V00901,A9999,Baixa,Cloud Security,04/01/2026,29/03/2026,Corrigida,5.8
1300,V01301,A9999,Baixa,Cloud Security,21/07/2026,23/09/2026,Corrigida,7.5
1700,V01701,A9999,Crítica,Cloud Security,08/04/2026,31/05/2026,Corrigida,5.8


Foram identificadas 5 vulnerabilidades associadas ao ativo `A9999`, que não possui correspondência na base de ativos.

Esses registros serão removidos da camada Silver, pois não possuem uma referência válida para um ativo existente.

## 6. Validação Final

In [11]:
ativos_tratados.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ativo_id      200 non-null    str  
 1   sistema       200 non-null    str  
 2   criticidade   196 non-null    str  
 3   dominio       200 non-null    str  
 4   ambiente      198 non-null    str  
 5   localidade    200 non-null    str  
 6   status_ativo  200 non-null    str  
dtypes: str(7)
memory usage: 22.5 KB


Foram identificados 4 ativos sem classificação de criticidade. Como não há informação suficiente para determinar a criticidade correta, os valores foram mantidos como nulos (`NaN`).

Também foram identificados 2 ativos sem informação de ambiente. Como não há informação suficiente para determinar o ambiente correto, os valores foram mantidos como nulos (`NaN`).

In [12]:
vulnerabilidades_tratadas.info()

<class 'pandas.DataFrame'>
Index: 1995 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   vuln_id        1995 non-null   str           
 1   ativo_id       1995 non-null   str           
 2   severidade     1969 non-null   str           
 3   origem         1995 non-null   str           
 4   data_abertura  1973 non-null   datetime64[us]
 5   data_correcao  466 non-null    datetime64[us]
 6   status         1995 non-null   str           
 7   cvss_score     1995 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 207.8 KB


Os 26 registros sem `severidade` foram mantidos como nulos, pois não há informação suficiente para determinar a classificação correta.

Os 22 registros com datas de abertura inválidas foram mantidos como `NaT`, evitando imputações sem respaldo.

Os valores ausentes em `data_correcao` foram mantidos como `NaT`, pois representam vulnerabilidades sem data de correção registrada.



## 6. Salvando os data frames em csv

In [13]:
ativos_tratados.to_parquet(
    '../data/silver/ativos.parquet',
    index=False
)

vulnerabilidades_tratadas.to_parquet(
    '../data/silver/vulnerabilidades.parquet',
    index=False
)